# Purpose

## Estimate the relationship between net domestic migration and county-level YoY home value growth during the 2021–2022 boom period.

# Imports

In [1]:
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

# Load Processed Dataset

In [2]:
PROJECT_ROOT = Path().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PROCESSED / "merged_fl_county_year_features.csv")
df.head(10)

,fips,RegionName,year,zhvi_annual,STNAME,CTYNAME,domestic_mig,population,rdomestic_mig,yoy_growth,boom_growth_2020_2022,cooling_delta_2024_vs_2022,yoy_volatility
0,12001,Alachua County,2020,213860.291033,Florida,Alachua County,1315,279765,NaN,NaN,0.319704,-0.125115,0.064152
1,12001,Alachua County,2021,244364.099887,Florida,Alachua County,1142,281710,4.067857,0.142634,0.319704,-0.125115,0.064152
2,12001,Alachua County,2022,282232.292655,Florida,Alachua County,1268,285241,4.473050,0.154966,0.319704,-0.125115,0.064152
3,12001,Alachua County,2023,295677.948240,Florida,Alachua County,408,288962,1.421100,0.047640,0.319704,-0.125115,0.064152
4,12001,Alachua County,2024,304504.265219,Florida,Alachua County,-1141,291782,-3.929442,0.029851,0.319704,-0.125115,0.064152
5,12003,Baker County,2020,214588.831681,Florida,Baker County,-141,28122,NaN,NaN,0.369415,-0.153888,0.094209
6,12003,Baker County,2021,250077.267178,Florida,Baker County,216,28378,7.646018,0.165379,0.369415,-0.153888,0.094209
7,12003,Baker County,2022,293861.137166,Florida,Baker County,-657,27781,-23.397853,0.175081,0.369415,-0.153888,0.094209
8,12003,Baker County,2023,292472.265114,Florida,Baker County,673,28542,23.897875,-0.004726,0.369415,-0.153888,0.094209
9,12003,Baker County,2024,298670.668188,Florida,Baker County,709,29325,24.504467,0.021193,0.369415,-0.153888,0.094209


# Boom Period Regression

## Subset to Boom Period

In [3]:
df_boom = df[df["year"].isin([2021, 2022])].copy()

In [4]:
df_boom.shape

(134, 13)

## Define Variables

In [5]:
# Dependent variable (what we are trying to explain)
Y = df_boom["yoy_growth"]

In [6]:
# Independent variable (what we think explains growth)
X = df_boom["rdomestic_mig"]

## Add Constant

In [7]:
# Add a constant column of 1s so we can estimate an intercept
# This allows the model to estimate baseline growth when migration = 0
X = sm.add_constant(X)

In [8]:
X.head()

,const,rdomestic_mig
1,1.0,4.067857
2,1.0,4.473050
6,1.0,7.646018
7,1.0,-23.397853
11,1.0,30.607951


## Run OLS (Ordinary Least Squares)

In [17]:
model = sm.OLS(Y, X) # Create the OLS regression model
results_boom = model.fit() # Fit the model (estimate intercept and slope)

print(results_boom.summary())

                            OLS Regression Results                            
Dep. Variable:             yoy_growth   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.422
Date:                Tue, 03 Mar 2026   Prob (F-statistic):              0.235
Time:                        15:42:14   Log-Likelihood:                 284.95
No. Observations:                 134   AIC:                            -565.9
Df Residuals:                     132   BIC:                            -560.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0337      0.004      9.101

# Cooling Period Regression

## Subset to Cooling Period

In [10]:
df_cooling = df[df["year"].isin([2023, 2024])].copy()

## Define Variables

In [13]:
# Dependent variable (what we are trying to explain)
Y = df_cooling["yoy_growth"]

In [14]:
# Independent variable (what we think explains growth)
X = df_cooling["rdomestic_mig"]

## Add constant

In [15]:
X = sm.add_constant(X)

## Run OLS

In [16]:
results_cooling = sm.OLS(Y, X).fit()
print(results_cooling.summary())

                            OLS Regression Results                            
Dep. Variable:             yoy_growth   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.422
Date:                Tue, 03 Mar 2026   Prob (F-statistic):              0.235
Time:                        15:41:45   Log-Likelihood:                 284.95
No. Observations:                 134   AIC:                            -565.9
Df Residuals:                     132   BIC:                            -560.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0337      0.004      9.101